In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import mne
from sklearn.linear_model import Ridge

import warnings
warnings.filterwarnings('ignore')
from scipy.signal import filtfilt
from joblib import Parallel, delayed
from tqdm import tqdm
from mne.stats import permutation_cluster_test, permutation_cluster_1samp_test


# Import plot config
from plot_config import set_plot_style
set_plot_style()

# Import all analysis functions from utility module
from mv_regression_utils import (
    Subject,
    load_subject_data,
    fit_encoding_model,
    compute_time_resolved_reactivation,
    compute_subject_reactivation,
    run_cluster_test_per_roi,
    run_encoding_weight_cluster_test,
    plot_encoding_weight_clusters,
    load_trial_level,
    load_all_trial_level_per_roi,
    run_2sample_cluster_test_per_roi,
    run_subject_level_1sample_test,
    run_subject_wise_2sample_cluster_test,
    compute_all_subjects_cluster_statistics,
    plot_cluster_statistics_bar_chart,
    run_per_electrode_2sample_cluster_test,
    run_per_electrode_2sample_cluster_test_with_t_obs,
    compute_per_electrode_cluster_counts,
    plot_electrode_cluster_counts,
    multinomial_regression,
    _encode_memory_outcome,
    _fit_pixel_regression,
    _process_pixel,
    _process_single_subject,
    run_reactivation_pipeline,
    compute_subject_lrt_map,
    compute_all_subjects_lrt_maps
)

from mv_granger_utils import (
    load_behavior_tables,
    preprocess_learn_df,
    preprocess_combined_df
)


In [ ]:
# =============================
# Analysis configuration
# =============================

USE_SURROGATES = True   # Enable surrogate analysis for z-scoring
N_SURROGATES   = 250     # Number of surrogates for null distribution
RIDGE_ALPHA = 1

# ROIs to analyze
ROIS = ['ACC']

# Dependent variables to analyze
DEPENDENT_VARS = ['zrpe']

# Reactivation computation mode
ELECTRODE_LEVEL = False  # If True, compute reactivation at electrode level (4D output)
                         # If False, compute vector-level reactivation (3D output, default)

# Analysis parameters
downsample_factor = 5
win_ms = 200
hfa_freq_range = (70., 201.)
min_trials = 10

# Cluster test parameters
N_PERMUTATIONS = 1024
CLUSTER_SEED = 42

# Parallelization parameters
N_JOBS = 4  # Number of parallel jobs (-1 = use all CPUs) - Reduced from 8 to prevent memory crashes
PARALLEL_SUBJECTS = True  # Parallelize across subjects (recommended)
PARALLEL_SURROGATES = True  # Parallelize surrogate computation (recommended)

# Trial-level saving (z-scored reactivation)
SAVE_TRIAL_LEVEL = True
TRIAL_SAVE_DIR = '/Volumes/T7 Shield/scratch/MemoryBandit/RPE_Reactivation/trial_level'
TRIAL_SAVE_DTYPE = np.float32
OVERWRITE_TRIAL_LEVEL = True

In [ ]:
# Setup paths
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load demographics data
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Filter out bad patients
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

print(f"Available subjects: {subj_df.MSSMCode.values}")


In [ ]:
# Load + preprocess behavior
learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)


In [ ]:
sns.kdeplot(learn_df[learn_df.reward==1].rpe)

Load one subject's data

In [ ]:
subject='MS036'
base_dir=base_dir
load_dir=load_dir
roi='dmPFC'
downsample_factor=5
win_ms = 200
hfa_freq_range = (70., 201.)
min_trials = 10
verbose = True
dependent_var = 'preward'

data = load_subject_data(subject, base_dir, load_dir, roi,
                            downsample_factor, win_ms, hfa_freq_range, min_trials, verbose, dependent_var)

In [ ]:
fb_hfa   = data['feedback_hfa']   # trials × electrodes × timepoints (see load_subject_data)
cue_hfa  = data['cue_hfa']
dependent_var   = data['dependent_var']
memory   = data['memory_outcomes']         # 1 = hit, 0 = miss

# -----------------------------
# 1) Fit RPE encoding model
encoding_array = np.zeros((fb_hfa.shape[-1], fb_hfa.shape[1]))
if ELECTRODE_LEVEL:
    reactivation_array = np.zeros((fb_hfa.shape[-1], cue_hfa.shape[-1], fb_hfa.shape[1], cue_hfa.shape[0]))
else:
    reactivation_array = np.zeros((fb_hfa.shape[-1], cue_hfa.shape[-1], cue_hfa.shape[0]))

for t in range(fb_hfa.shape[-1]):
    X = fb_hfa[:, :, t]
    w_real, b_real = fit_encoding_model(
        X, dependent_var, alpha=RIDGE_ALPHA
    )
    encoding_array[t] = w_real

    reactivation = compute_time_resolved_reactivation(cue_hfa, w_real, b_real, electrode_level=ELECTRODE_LEVEL)
    if ELECTRODE_LEVEL:
        reactivation_array[t] = np.transpose(reactivation, (2, 1, 0))
    else:
        reactivation_array[t, :, :] = reactivation.T

    # copmute activation pattern from multivariate regression
    # 1. Compute the empirical covariance matrix of your brain data
    # rowvar=False because columns are our electrode channels
    sigma_x = np.cov(X, rowvar=False)

    # 2. Compute the activation pattern A
    # A = Sigma_x @ W
    activation_pattern = sigma_x @ w_real

    # plt.imshow(w_real[np.newaxis, :])


Below is the encoding vector for ACC (7 electrodes) for one patient at one timepoint

In [ ]:
plt.imshow(w_real[np.newaxis, :])

In [ ]:
plt.imshow(activation_pattern[np.newaxis, :])

In [ ]:
from scipy.stats import pearsonr
pearsonr(activation_pattern, w_real)

Below is the encoding vectors for ACC for one patient at all timepoints

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(20, 10))
ax.imshow(encoding_array.T)

In [ ]:
# =============================================================================
# STEP 1: Initialize Subject objects ONCE (does all preprocessing)
# This loads TFR files and preprocesses behavioral data for all subjects
# =============================================================================
print(f"\n{'='*70}")
print("INITIALIZING SUBJECTS (Loading TFR files and preprocessing behavioral data)")
print(f"{'='*70}")
print("This step loads TFR files and preprocesses behavioral data once per subject.")
print("Subsequent analyses will reuse this preprocessed data efficiently.")
print(f"{'='*70}")

subject_objects = {}
subjects = subj_df.MSSMCode.values

for subject in subjects:
    try:
        print(f"\nInitializing {subject}...")
        subject_obj = Subject(
            subject=subject,
            base_dir=base_dir,
            load_dir=load_dir,
            min_trials=min_trials,
            verbose=True,
            neural_cache_dir=None  # Uses default: {base_dir}/.../{subject}/scratch/neural_cache/
            # Or specify custom: neural_cache_dir='/path/to/cache'
        )
        subject_objects[subject] = subject_obj
        print(f"  ✓ {subject} initialized successfully!")
    except Exception as e:
        print(f"  ✗ {subject} failed to initialize: {e}")
        subject_objects[subject] = None
        import traceback
        traceback.print_exc()

print(f"\n{'='*70}")
n_initialized = len([s for s in subject_objects.values() if s is not None])
print(f"Initialized {n_initialized}/{len(subjects)} subjects")
print(f"{'='*70}")
print("\nSubject objects are now ready for efficient processing!")
print("Neural data (TFR files) loaded once per subject.")
print("Behavioral preprocessing completed for all subjects.")
print("Processed neural data will be cached to disk for future sessions.")

In [ ]:
# # =============================================================================
# # SINGLE PATIENT ELECTRODE-LEVEL ANALYSIS: Run for one patient first
# # =============================================================================
# # Uses the same run_reactivation_pipeline as the main analysis (one subject).

# # Select ONE patient to analyze
# TEST_PATIENT = "MS036"
# # subj_df.MSSMCode.values[0]  # Change this to test different patients
# print(f"\n{'='*70}")
# print(f"RUNNING ELECTRODE-LEVEL ANALYSIS FOR SINGLE PATIENT: {TEST_PATIENT}")
# print(f"{'='*70}")
# print(f"ELECTRODE_LEVEL = {ELECTRODE_LEVEL}")
# print(f"{'='*70}\n")

# surrogate_n_jobs = N_JOBS if PARALLEL_SURROGATES else 1
# pipeline_results = run_reactivation_pipeline(
#     subjects=[TEST_PATIENT],
#     rois=ROIS,
#     dependent_vars=DEPENDENT_VARS,
#     base_dir=base_dir,
#     load_dir=load_dir,
#     downsample_factor=downsample_factor,
#     win_ms=win_ms,
#     hfa_freq_range=hfa_freq_range,
#     min_trials=min_trials,
#     n_surrogates=N_SURROGATES,
#     use_surrogates=USE_SURROGATES,
#     ridge_alpha=RIDGE_ALPHA,
#     seed=CLUSTER_SEED,
#     n_jobs=surrogate_n_jobs,
#     save_trial_level=SAVE_TRIAL_LEVEL,
#     trial_save_dir=TRIAL_SAVE_DIR,
#     overwrite_trial_level=OVERWRITE_TRIAL_LEVEL,
#     trial_save_dtype=TRIAL_SAVE_DTYPE,
#     electrode_level=ELECTRODE_LEVEL,
#     surrogate_chunk_size=None,
#     n_jobs_outer=1,
#     verbose=True,
# )

# # Convert to single_patient_results[roi][dependent_var] = dict or None (same shape as before)
# single_patient_results = {}
# for roi in ROIS:
#     single_patient_results[roi] = {}
#     for dependent_var in DEPENDENT_VARS:
#         lst = pipeline_results.get(roi, {}).get(dependent_var, {})
#         subject_list = lst.get('subject_list', [])
#         if len(subject_list) == 0:
#             single_patient_results[roi][dependent_var] = None
#         else:
#             single_patient_results[roi][dependent_var] = {
#                 'feedback_times': lst['feedback_times_list'][0],
#                 'cue_times': lst['cue_times_list'][0],
#                 'weights': lst['weights_list'][0],
#                 'null_weights': lst['null_weights_list'][0],
#                 'memory_outcomes': lst['memory_outcomes'][0],
#                 'subject': lst['subject_list'][0],
#             }

# # Verify saved data per ROI/dependent_var
# for roi in ROIS:
#     for dependent_var in DEPENDENT_VARS:
#         if single_patient_results[roi][dependent_var] is None:
#             continue
#         try:
#             z_reactivation, fb_times, cue_times, meta = load_trial_level(
#                 roi, TEST_PATIENT, TRIAL_SAVE_DIR, dependent_var=dependent_var
#             )
#             if z_reactivation.ndim == 4:
#                 print(f"  {roi} / {dependent_var}: ✓ Electrode-level data saved: shape {z_reactivation.shape}")
#             else:
#                 print(f"  {roi} / {dependent_var}: ⚠ Vector-level data saved: shape {z_reactivation.shape}")
#         except Exception as e:
#             print(f"  {roi} / {dependent_var}: ⚠ Could not verify saved data: {e}")

# print(f"\n{'='*70}")
# print(f"SINGLE PATIENT ANALYSIS COMPLETE: {TEST_PATIENT}")
# print(f"{'='*70}")
# print(f"\nSummary:")
# for roi in ROIS:
#     for dependent_var in DEPENDENT_VARS:
#         if single_patient_results[roi][dependent_var] is not None:
#             print(f"  {roi} - {dependent_var}: ✓ Processed")
#         else:
#             print(f"  {roi} - {dependent_var}: ✗ Skipped/Failed")

In [ ]:
# =============================================================================
# MAIN ANALYSIS: Subject-first pipeline (one TFR load per subject, all ROIs/vars)
# Uses run_reactivation_pipeline from mv_regression_utils
# =============================================================================

subjects = subj_df.MSSMCode.values

surrogate_n_jobs = N_JOBS if PARALLEL_SURROGATES else 1
results_per_roi_per_var = run_reactivation_pipeline(
    subjects=list(subjects),
    rois=ROIS,
    dependent_vars=DEPENDENT_VARS,
    base_dir=base_dir,
    load_dir=load_dir,
    downsample_factor=downsample_factor,
    win_ms=win_ms,
    hfa_freq_range=hfa_freq_range,
    min_trials=min_trials,
    n_surrogates=N_SURROGATES,
    use_surrogates=USE_SURROGATES,
    ridge_alpha=RIDGE_ALPHA,
    seed=CLUSTER_SEED,
    n_jobs=surrogate_n_jobs,
    save_trial_level=SAVE_TRIAL_LEVEL,
    trial_save_dir=TRIAL_SAVE_DIR,
    overwrite_trial_level=OVERWRITE_TRIAL_LEVEL,
    trial_save_dtype=TRIAL_SAVE_DTYPE,
    electrode_level=ELECTRODE_LEVEL,
    surrogate_chunk_size=None,
    n_jobs_outer=1,
    verbose=True,
)

for roi in ROIS:
    for dependent_var in DEPENDENT_VARS:
        if roi in results_per_roi_per_var and dependent_var in results_per_roi_per_var[roi]:
            n_subj = len(results_per_roi_per_var[roi][dependent_var]['subject_list'])
            print(f"ROI {roi}, {dependent_var}: {n_subj} subjects processed")

In [ ]:
# # =============================================================================
# # ALL-SUBJECT ELECTRODE-LEVEL: Aggregate significant electrodes (hits vs misses)
# # and plot group + example heatmaps of hits-misses difference
# # =============================================================================

# # Configuration
# DEPENDENT_VAR_ELECTRODE = DEPENDENT_VARS[0]
# subjects_electrode = subj_df.MSSMCode.values
# N_EXAMPLE_SUBJECTS = 2

# print(f"\n{'='*70}")
# print(f"ALL-SUBJECT ELECTRODE-LEVEL: Hits vs Misses (cluster test + heatmaps)")
# print(f"{'='*70}")
# print(f"  Dependent var: {DEPENDENT_VAR_ELECTRODE}, Subjects: {len(subjects_electrode)}")

# # Per-subject, per-ROI: load data, run cluster test, compute hits-misses diff per electrode
# results = {roi: {} for roi in ROIS}
# for subj_idx, subject in enumerate(subjects_electrode):
#     if (subj_idx + 1) % 5 == 0 or subj_idx == 0:
#         print(f"  Processing subject {subj_idx + 1}/{len(subjects_electrode)}: {subject}")
#     for roi in ROIS:
#         try:
#             z_reactivation, fb_times, cue_times, meta = load_trial_level(
#                 roi, subject, TRIAL_SAVE_DIR, dependent_var=DEPENDENT_VAR_ELECTRODE
#             )
#             subj_df_filtered = combined_df[
#                 (combined_df.participant == subject) & (combined_df.condition == 'Day1')
#             ]
#             if 'memory_outcome' not in subj_df_filtered.columns:
#                 subj_df_filtered = subj_df_filtered.copy()
#                 subj_df_filtered['memory_outcome'] = subj_df_filtered[
#                     ['hits', 'misses', 'correct_rejections', 'false_alarms']
#                 ].idxmax(1)
#             memory_outcomes = subj_df_filtered['memory_outcome'].values
#             if z_reactivation.ndim == 4:
#                 n_trials_reactivation = z_reactivation.shape[3]
#                 n_electrodes = z_reactivation.shape[2]
#             else:
#                 n_trials_reactivation = z_reactivation.shape[2]
#                 n_electrodes = 1
#                 z_reactivation = z_reactivation[:, :, np.newaxis, :]
#             if len(memory_outcomes) != n_trials_reactivation:
#                 min_len = min(len(memory_outcomes), n_trials_reactivation)
#                 memory_outcomes = memory_outcomes[:min_len]
#                 z_reactivation = z_reactivation[:, :, :, :min_len]
#             if z_reactivation.ndim != 4:
#                 continue
#             cluster_counts = run_per_electrode_2sample_cluster_test(
#                 z_reactivation=z_reactivation,
#                 memory_outcomes=memory_outcomes,
#                 n_permutations=N_PERMUTATIONS,
#                 cluster_seed=CLUSTER_SEED,
#                 verbose=False
#             )
#             sig_mask = cluster_counts > 0
#             hits_mask = memory_outcomes == 'hits'
#             misses_mask = memory_outcomes == 'misses'
#             if np.sum(hits_mask) < 2 or np.sum(misses_mask) < 2:
#                 continue
#             hits_misses_diff = np.zeros((z_reactivation.shape[0], z_reactivation.shape[1], z_reactivation.shape[2]))
#             for e in range(z_reactivation.shape[2]):
#                 hits_m = np.mean(z_reactivation[:, :, e, :][:, :, hits_mask], axis=2)
#                 miss_m = np.mean(z_reactivation[:, :, e, :][:, :, misses_mask], axis=2)
#                 hits_misses_diff[:, :, e] = hits_m - miss_m
#             results[roi][subject] = {
#                 'cluster_counts': cluster_counts,
#                 'sig_mask': sig_mask,
#                 'hits_misses_diff': hits_misses_diff,
#                 'n_electrodes': z_reactivation.shape[2],
#                 'fb_times': fb_times,
#                 'cue_times': cue_times,
#             }
#         except Exception:
#             continue

# # Aggregate: group heatmap (mean over subjects of mean over sig electrodes) and counts
# n_subjects_with_sig = {}
# group_heatmap = {}
# example_subjects = {}
# for roi in ROIS:
#     n_subjects_with_sig[roi] = sum(1 for s in results[roi] if np.any(results[roi][s]['sig_mask']))
#     subject_maps = []
#     for subject in results[roi]:
#         d = results[roi][subject]
#         if not np.any(d['sig_mask']):
#             continue
#         mean_sig = np.nanmean(d['hits_misses_diff'][:, :, d['sig_mask']], axis=2)
#         subject_maps.append(mean_sig)
#     if len(subject_maps) > 0:
#         group_heatmap[roi] = np.nanmean(np.stack(subject_maps, axis=0), axis=0)
#     else:
#         group_heatmap[roi] = None
#     subjects_sorted = sorted(
#         [s for s in results[roi] if np.any(results[roi][s]['sig_mask'])],
#         key=lambda s: np.sum(results[roi][s]['sig_mask']),
#         reverse=True
#     )
#     example_subjects[roi] = subjects_sorted[:N_EXAMPLE_SUBJECTS]

# # Plot: one row per ROI; columns = Group heatmap, Example 1, Example 2
# rois_plot = [r for r in ROIS if r in results and (group_heatmap.get(r) is not None or len(example_subjects.get(r, [])) > 0)]
# if len(rois_plot) == 0:
#     print("No ROI had any subject with significant electrodes.")
# else:
#     n_cols = 1 + N_EXAMPLE_SUBJECTS
#     fig, axes = plt.subplots(len(rois_plot), n_cols, figsize=(5 * n_cols, 4 * len(rois_plot)))
#     if len(rois_plot) == 1:
#         axes = axes.reshape(1, -1)
#     all_maps = []
#     for roi in rois_plot:
#         if group_heatmap.get(roi) is not None:
#             all_maps.append(group_heatmap[roi])
#         for s in example_subjects.get(roi, []):
#             m = np.nanmean(results[roi][s]['hits_misses_diff'][:, :, results[roi][s]['sig_mask']], axis=2)
#             all_maps.append(m)
#     vmax = np.nanmax(np.abs(all_maps)) if all_maps else 1.0
#     vmin = -vmax
#     for row_idx, roi in enumerate(rois_plot):
#         ax = axes[row_idx, 0]
#         if group_heatmap.get(roi) is not None:
#             first_subj = next(iter(results[roi]))
#             fb = results[roi][first_subj]['fb_times']
#             cue = results[roi][first_subj]['cue_times']
#             im = ax.imshow(group_heatmap[roi], aspect='auto', origin='lower', cmap='RdBu_r', vmin=vmin, vmax=vmax,
#                            extent=[cue[0], cue[-1], fb[0], fb[-1]])
#             ax.set_title(f'{roi}: Group (sig electrodes only)\nn={n_subjects_with_sig[roi]} subjects')
#         else:
#             ax.text(0.5, 0.5, 'No significant electrodes', ha='center', va='center')
#             ax.set_title(f'{roi}: Group')
#         ax.set_xlabel('Cue Time (s)')
#         ax.set_ylabel('Feedback Time (s)')
#         if row_idx == 0 and group_heatmap.get(roi) is not None:
#             plt.colorbar(im, ax=ax, label='Hits - Misses (Z)')
#         for ex_idx, subject in enumerate(example_subjects.get(roi, [])[:N_EXAMPLE_SUBJECTS]):
#             ax = axes[row_idx, ex_idx + 1]
#             d = results[roi][subject]
#             m = np.nanmean(d['hits_misses_diff'][:, :, d['sig_mask']], axis=2)
#             im = ax.imshow(m, aspect='auto', origin='lower', cmap='RdBu_r', vmin=vmin, vmax=vmax,
#                            extent=[d['cue_times'][0], d['cue_times'][-1], d['fb_times'][0], d['fb_times'][-1]])
#             n_sig = np.sum(d['sig_mask'])
#             ax.set_title(f'{roi}: {subject}\n{n_sig} sig electrodes')
#             ax.set_xlabel('Cue Time (s)')
#             ax.set_ylabel('Feedback Time (s)')
#             if row_idx == 0:
#                 plt.colorbar(im, ax=ax, label='Hits - Misses (Z)')
#     plt.suptitle(f'All-Subject Electrode-Level: Hits - Misses (significant electrodes only)', fontsize=14, y=1.02)
#     plt.tight_layout()
#     plt.show()

#     print(f"\n{'='*70}")
#     print("Summary (subjects with at least one significant electrode):")
#     for roi in ROIS:
#         n_sig = n_subjects_with_sig.get(roi, 0)
#         n_subj_roi = len(results.get(roi, {}))
#         counts = [np.sum(results[roi][s]['sig_mask']) for s in results.get(roi, {}) if np.any(results[roi][s]['sig_mask'])]
#         mean_sig = np.mean(counts) if counts else 0
#         max_sig = np.max(counts) if counts else 0
#         print(f"  {roi}: {n_sig}/{n_subj_roi} subjects with >=1 sig electrode; mean sig electrodes (those subjects): {mean_sig:.1f}, max: {max_sig}")



In [ ]:
roi = 'ACC'

# Loop over dependent variables
all_z_dict = {}
trunc_combined_df_dict = {}
rearanged_z_reactivation_dict = {}

for dependent_var in DEPENDENT_VARS:
    print(f"\nLoading data for dependent_var: {dependent_var}")
    all_z = []
    trunc_combined_df = []
    
    for subj in subj_df.MSSMCode.values:
        try:
            z_reactivation, fb_times, cue_times, meta = load_trial_level(
                roi, subj, TRIAL_SAVE_DIR, dependent_var=dependent_var
            )
        except:
            continue
        all_z.append(z_reactivation.mean(axis=2))
        trunc_combined_df.append(combined_df[(combined_df.participant==subj) & (combined_df.condition=='Day1')])
    
    if len(all_z) == 0:
        print(f"Warning: No data loaded for {dependent_var}")
        continue
    
    all_z = np.concatenate(all_z, axis=-1)
    rearanged_z_reactivation = np.transpose(all_z, (2, 0, 1))
    trunc_combined_df = pd.concat(trunc_combined_df)
    
    if len(trunc_combined_df) != rearanged_z_reactivation.shape[0]:
        print(f"WARNING: {dependent_var} has {len(trunc_combined_df)} trials, but {rearanged_z_reactivation.shape[0]} z-reactivation values")
    
    # Store in dictionaries
    all_z_dict[dependent_var] = all_z
    trunc_combined_df_dict[dependent_var] = trunc_combined_df
    rearanged_z_reactivation_dict[dependent_var] = rearanged_z_reactivation
    
    # Create memory_outcome column if it doesn't exist
    if 'memory_outcome' not in trunc_combined_df.columns:
        trunc_combined_df['memory_outcome'] = trunc_combined_df[['hits', 'misses', 'correct_rejections', 
                                                                 'false_alarms']].idxmax(1)
    
    # Store reactivation arrays by outcome (for reference)
    reactivation_hits = rearanged_z_reactivation[trunc_combined_df.memory_outcome=='hits']
    reactivation_misses = rearanged_z_reactivation[trunc_combined_df.memory_outcome=='misses']
    reactivation_false_alarms = rearanged_z_reactivation[trunc_combined_df.memory_outcome=='false_alarms']
    reactivation_correct_rejections = rearanged_z_reactivation[trunc_combined_df.memory_outcome=='correct_rejections']
    
    print(f"{dependent_var}: Loaded {len(trunc_combined_df)} trials, shape {rearanged_z_reactivation.shape}")

print(f"\nData loading complete for {len(all_z_dict)} dependent variables")



In [ ]:
# =============================================================================
# OPTIMIZED: Pre-process data once and parallelize regression
# Helper functions are imported from mv_regression_utils
# =============================================================================

# Initialize result dictionaries for each dependent variable
misses_results_dict = {}
correct_rejections_results_dict = {}
false_alarms_results_dict = {}
lrt_results_dict = {}
aic_results_dict = {}

# Loop over dependent variables
for dependent_var in DEPENDENT_VARS:
    if dependent_var not in rearanged_z_reactivation_dict:
        print(f"Skipping {dependent_var}: no data available")
        continue
    
    print(f"\n{'='*70}")
    print(f"Processing {dependent_var}")
    print(f"{'='*70}")
    
    rearanged_z_reactivation = rearanged_z_reactivation_dict[dependent_var]
    trunc_combined_df = trunc_combined_df_dict[dependent_var]
    
    # Pre-process dataframe once: encode memory_outcome and create valid mask
    # Encode response variable once
    y_encoded = _encode_memory_outcome(trunc_combined_df['memory_outcome'].values)
    
    # Create valid mask (non-NaN in memory_outcome)
    valid_mask = ~np.isnan(y_encoded)
    
    # Create all pixel combinations
    n_i, n_j = rearanged_z_reactivation.shape[1], rearanged_z_reactivation.shape[2]
    pixel_combinations = [(i, j) for i in range(n_i) for j in range(n_j)]
    
    # Initialize result arrays
    misses_results = np.zeros((n_i, n_j))
    correct_rejections_results = np.zeros((n_i, n_j))
    false_alarms_results = np.zeros((n_i, n_j))
    lrt_results = np.zeros((n_i, n_j))
    aic_results = np.zeros((n_i, n_j))
    
    # Process all pixels in parallel
    print(f"Processing {len(pixel_combinations)} pixels in parallel...")
    results = Parallel(n_jobs=-1, verbose=1)(
        delayed(_process_pixel)(ij, rearanged_z_reactivation, y_encoded, valid_mask) 
        for ij in pixel_combinations
    )
    
    # Reshape results into arrays
    for idx, (i, j) in enumerate(pixel_combinations):
        misses_coef, cr_coef, fa_coef, lrt, aic = results[idx]
        misses_results[i, j] = misses_coef
        correct_rejections_results[i, j] = cr_coef
        false_alarms_results[i, j] = fa_coef
        lrt_results[i, j] = lrt
        aic_results[i, j] = aic
    
    # Store results in dictionaries
    misses_results_dict[dependent_var] = misses_results
    correct_rejections_results_dict[dependent_var] = correct_rejections_results
    false_alarms_results_dict[dependent_var] = false_alarms_results
    lrt_results_dict[dependent_var] = lrt_results
    aic_results_dict[dependent_var] = aic_results
    
    print(f"{dependent_var}: Parallel processing complete!")
    print(f"  AIC shape: {aic_results.shape}")
    print(f"  AIC range: [{np.nanmin(aic_results):.2f}, {np.nanmax(aic_results):.2f}]")

print(f"\n{'='*70}")
print("All dependent variables processed!")
print(f"{'='*70}")


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

# Panel 1: Mean LRT map across subjects
ax.imshow(lrt_results_dict['zrpe'],
aspect='auto', origin='lower',
                    cmap='RdBu_r')
ax.set_xlabel('Cue Time')
ax.set_ylabel('Feedback Time')
# plt.colorbar(im1, ax=axes[0], label='aic difference')

In [ ]:
# =============================================================================
# Subject-by-Subject LRT Analysis with MNE Permutation Test
# =============================================================================
# This cell computes LRT maps for each subject separately, then runs
# a group-level permutation cluster test on the resulting 3D array.

# Set ROI to analyze
roi = 'HPC'  # Change this to analyze different ROIs

# Store results for each dependent variable
lrt_maps_dict = {}
valid_subjects_dict = {}
feedback_times_dict = {}
cue_times_dict = {}

# Loop over dependent variables
for dependent_var in DEPENDENT_VARS:
    print(f"\n{'='*70}")
    print(f"COMPUTING LRT MAPS FOR ROI: {roi}, DEPENDENT VAR: {dependent_var}")
    print(f"{'='*70}")
    
    # Compute LRT maps for all subjects (subject-by-subject)
    # This returns a 3D array: (n_subj, n_feedback_t, n_cue_t)
    lrt_maps, valid_subjects, feedback_times, cue_times = compute_all_subjects_lrt_maps(
        roi=roi,
        subjects=subj_df.MSSMCode.values,
        combined_df=combined_df,
        trial_save_dir=TRIAL_SAVE_DIR,
        n_jobs=N_JOBS,
        verbose=True,
        dependent_var=dependent_var
    )
    
    if lrt_maps is not None:
        lrt_maps_dict[dependent_var] = lrt_maps
        valid_subjects_dict[dependent_var] = valid_subjects
        feedback_times_dict[dependent_var] = feedback_times
        cue_times_dict[dependent_var] = cue_times

# Process each dependent variable
for dependent_var in DEPENDENT_VARS:
    if dependent_var not in lrt_maps_dict:
        print(f"\nSkipping {dependent_var}: no LRT maps computed")
        continue
    
    lrt_maps = lrt_maps_dict[dependent_var]
    valid_subjects = valid_subjects_dict[dependent_var]
    feedback_times = feedback_times_dict[dependent_var]
    cue_times = cue_times_dict[dependent_var]
    
    print(f"\n{'='*70}")
    print(f"LRT MAPS COMPUTED FOR ROI: {roi}, DEPENDENT VAR: {dependent_var}")
    print(f"{'='*70}")
    print(f"Shape: {lrt_maps.shape} (n_subjects, n_feedback_t, n_cue_t)")
    print(f"Valid subjects: {len(valid_subjects)}")
    print(f"Subjects: {valid_subjects}")
    
    # Run MNE permutation cluster test on the 3D LRT array
    print(f"\n{'='*70}")
    print(f"RUNNING MNE PERMUTATION CLUSTER TEST FOR {dependent_var}")
    print(f"{'='*70}")
    
    t_obs, clusters, cluster_pvals, h0 = permutation_cluster_1samp_test(
        lrt_maps,
        n_permutations=1000,
        tail=1,  # Two-tailed test
        n_jobs=-1,
        seed=42,
        verbose=False
    )
    
    # Display results
    print(f"\n{'='*70}")
    print(f"CLUSTER TEST RESULTS FOR ROI: {roi}, DEPENDENT VAR: {dependent_var}")
    print(f"{'='*70}")
    n_sig = np.sum(cluster_pvals <= 0.05) if len(cluster_pvals) > 0 else 0
    print(f"Found {len(clusters)} clusters, {n_sig} significant (p <= 0.05)")
    if len(cluster_pvals) > 0:
        print(f"Cluster p-values: {cluster_pvals}")
    
    # Plot results
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: Mean LRT map across subjects
    mean_lrt = np.mean(lrt_maps, axis=0)
    im1 = axes[0].imshow(mean_lrt, aspect='auto', origin='lower',
                        cmap='RdBu_r')
    axes[0].set_title(f'{roi} - {dependent_var}: Mean LRT Map\n(across {len(valid_subjects)} subjects)')
    axes[0].set_xlabel('Cue Time')
    axes[0].set_ylabel('Feedback Time')
    plt.colorbar(im1, ax=axes[0], label='LRT')
    
    # Panel 2: T-statistic map with significant clusters
    im2 = axes[1].imshow(t_obs, aspect='auto', origin='lower',
                        cmap='RdBu_r',)
    axes[1].set_title(f'{roi} - {dependent_var}: T-statistic Map\n(Cluster Test Results)')
    axes[1].set_xlabel('Cue Time')
    axes[1].set_ylabel('Feedback Time')
    plt.colorbar(im2, ax=axes[1], label='T-statistic')
    
    # Overlay significant clusters
    n_sig = 0
    if len(clusters) > 0:
        for cluster, pval in zip(clusters, cluster_pvals):
            if pval <= 0.05:
                n_sig += 1
                
                # Draw contour for this cluster
                cluster_mask = np.zeros_like(t_obs, dtype=bool)
                for a, b in zip(cluster[0], cluster[1]):
                    cluster_mask[a, b] = True

                axes[1].contour(cluster_mask.astype(float), levels=[0.5], 
                               colors='black', linewidths=1.5, linestyles='--')
    
    plt.tight_layout()
    plt.show()

print(f"\n{'='*70}")
print("LRT analysis complete for all dependent variables!")
print(f"{'='*70}")


In [ ]:
lrt_maps

In [ ]:
    t_obs, clusters, cluster_pvals, h0 = permutation_cluster_1samp_test(
        lrt_maps,
        n_permutations=1000,
        tail=1,  # Two-tailed test
        n_jobs=-1,
        seed=42,
        verbose=verbose
    )
    
    # Display results
    print(f"\n{'='*70}")
    print(f"CLUSTER TEST RESULTS FOR ROI: {roi}")
    print(f"{'='*70}")
    n_sig = np.sum(cluster_pvals <= 0.05) if len(cluster_pvals) > 0 else 0
    print(f"Found {len(clusters)} clusters, {n_sig} significant (p <= 0.05)")
    if len(cluster_pvals) > 0:
        print(f"Cluster p-values: {cluster_pvals}")
    
    # Plot results
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: Mean LRT map across subjects
    mean_lrt = np.mean(lrt_maps, axis=0)
    # vmax = np.max(np.abs(mean_lrt))
    # vmin = -vmax
    im1 = axes[0].imshow(mean_lrt, aspect='auto', origin='lower',
                        cmap='RdBu_r')
    axes[0].set_title(f'{roi}: Mean LRT Map\n(across {len(valid_subjects)} subjects)')
    axes[0].set_xlabel('Cue Time')
    axes[0].set_ylabel('Feedback Time')
    plt.colorbar(im1, ax=axes[0], label='LRT')
    
    # Panel 2: T-statistic map with significant clusters
    # vmax_t = np.max(np.abs(t_obs))
    # vmin_t = -vmax_t
    im2 = axes[1].imshow(t_obs, aspect='auto', origin='lower',
                        cmap='RdBu_r',)
    axes[1].set_title(f'{roi}: T-statistic Map\n(Cluster Test Results)')
    axes[1].set_xlabel('Cue Time')
    axes[1].set_ylabel('Feedback Time')
    plt.colorbar(im2, ax=axes[1], label='T-statistic')
    
    t_obs_mask = np.zeros_like(t_obs)
    # Overlay significant clusters
    if len(clusters) > 0:
        for cluster, pval in zip(clusters, cluster_pvals):
            if pval <= 0.05:
                n_sig += 1
                
                # Draw contour for this cluster
                cluster_mask = np.zeros_like(t_obs, dtype=bool)
                for a, b in zip(cluster[0], cluster[1]):
                    cluster_mask[a, b] = True

                ax.contour(cluster_mask.astype(float), levels=[0.5], colors='black',
                            linewidths=1.5, linestyles='--')
    

In [ ]:
lrt_maps

In [ ]:
cluster_pvals

In [ ]:
sns.heatmap(lrt_results, cmap='RdBu_r', vmax=30)

In [ ]:
result.summary()

In [ ]:
# plot all four reactivation arrays with labels
f, ax = plt.subplots(1, 3, figsize=(15, 5))
# sns.heatmap(np.mean(reactivation_hits, axis=0), cmap='RdBu_r', ax=ax[0])
sns.heatmap(-misses_results, cmap='RdBu_r', ax=ax[0], vmin=-0.3, vmax=0.3)
sns.heatmap(-correct_rejections_results, cmap='RdBu_r', ax=ax[1], vmin=-0.3, vmax=0.3)
sns.heatmap(-false_alarms_results, cmap='RdBu_r', ax=ax[2], vmin=-0.3, vmax=0.3)


In [ ]:
# Flatten the matrices into 1D vectors
vec1 = rearanged_z_reactivation[trunc_combined_df.memory_outcome=='hits'].mean(axis=0).ravel()
vec2 = rearanged_z_reactivation[trunc_combined_df.memory_outcome=='correct_rejections'].mean(axis=0).ravel()

# Compute Pearson's correlation coefficient
# pearsonr returns a tuple: (correlation_coefficient, p_value)
# We extract the first element, which is the coefficient
from scipy.stats import pearsonr
correlation_coefficient, p_value = pearsonr(vec1, vec2)

In [ ]:
# Plot t_obs heatmap with significant clusters outlined
f, ax = plt.subplots(1, 1, figsize=(2, 3))
sns.heatmap(T_obs, cmap='RdBu_r', ax=ax)
ax.invert_yaxis()

# Overlay significant clusters
n_sig = 0
# Plot T-statistic
vmax = np.max(np.abs(T_obs))
vmin = -vmax
im = ax.imshow(T_obs, aspect='auto', origin='lower',
                cmap='RdBu_r', vmin=vmin, vmax=vmax, interpolation='nearest')

t_obs_mask = np.zeros_like(T_obs)
# Overlay significant clusters
if len(clusters) > 0:
    for cluster, pval in zip(clusters, cluster_pvals):
        if pval <= 0.05:
            n_sig += 1
            
            # Draw contour for this cluster
            cluster_mask = np.zeros_like(T_obs, dtype=bool)
            for a, b in zip(cluster[0], cluster[1]):
                cluster_mask[a, b] = True

            ax.contour(cluster_mask.astype(float), levels=[0.5], colors='black',
                        linewidths=1.5, linestyles='--')
            
print(f"Found {len(clusters)} clusters, {n_sig} significant (p ≤ 0.05)")